# 00 — Data Setup

Baixa e consolida todas as releases do Anthropic Economic Index.

**Outputs:**
- `data/aei_all_releases.parquet` — todas as releases consolidadas
- `data/aei_aug2025.parquet` — Ago 2025 (enriched)
- `data/aei_feb2026.parquet` — Fev 2026 (mais recente)

**Metodologia do índice per capita:**  
`usage_per_capita_index = usage_pct / (pop_país / TOTAL_WAP)`  
onde `TOTAL_WAP` é a soma da `working_age_pop` dos países presentes no enriched de Ago 2025 — mesmo denominador que a Anthropic usa.

In [ ]:
%pip install -q huggingface_hub pandas pyarrow pycountry

In [ ]:
import pandas as pd
import pycountry
from pathlib import Path
from huggingface_hub import hf_hub_download, list_repo_files

ROOT     = Path('..').resolve()
DATA_DIR = ROOT / 'data'
DATA_DIR.mkdir(exist_ok=True)

REPO = 'Anthropic/EconomicIndex'

def dl(filename):
    return Path(hf_hub_download(repo_id=REPO, filename=filename,
                                repo_type='dataset', local_dir=DATA_DIR))

def to_iso3(x):
    if pd.isna(x): return x
    x = str(x).strip()
    if len(x) == 3: return x
    try: return pycountry.countries.get(alpha_2=x).alpha_3
    except: return x

print('Setup ok. ROOT:', ROOT)

## 1. Arquivos disponíveis no HuggingFace

In [ ]:
all_files = list(list_repo_files(REPO, repo_type='dataset'))
aei_files = [
    f for f in all_files
    if ('aei_enriched_claude_ai' in f or 'aei_raw_claude_ai' in f)
    and f.endswith('.csv')
]
print(f'{len(aei_files)} arquivo(s) AEI encontrado(s):')
for f in sorted(aei_files):
    print(f'  {f}')

## 2. Denominador correto para o índice per capita

In [ ]:
# Nomes dos países (World Bank)
POP_FILE = 'release_2025_09_15/data/input/working_age_pop_2024_country_raw.csv'
df_pop = pd.read_csv(dl(POP_FILE))
country_name_map = (
    df_pop[df_pop['countryiso3code'].str.match(r'^[A-Z]{3}$', na=False)]
    .set_index('countryiso3code')['country.value']
    .to_dict()
)

# Denominador fixo: soma working_age_pop do enriched de Ago 2025
# Este é o mesmo denominador que a Anthropic usa internamente
ENRICHED_FILE = 'release_2025_09_15/data/output/aei_enriched_claude_ai_2025-08-04_to_2025-08-11.csv'
df_enriched = pd.read_csv(dl(ENRICHED_FILE))

wap_rows = df_enriched[
    (df_enriched['geography'] == 'country') &
    (df_enriched['facet']     == 'country') &
    (df_enriched['variable']  == 'working_age_pop')
][['geo_id', 'value']].dropna()

wap_dict  = wap_rows.set_index('geo_id')['value'].to_dict()
TOTAL_WAP = sum(wap_dict.values())
pop_share = {iso3: pop / TOTAL_WAP for iso3, pop in wap_dict.items()}

# Supplementa com World Bank para países que aparecem nas raw mas não no enriched
df_pop_c = df_pop[
    df_pop['countryiso3code'].str.match(r'^[A-Z]{3}$', na=False) &
    df_pop['value'].notna() & (df_pop['value'] > 0)
]
for _, row in df_pop_c.iterrows():
    if row['countryiso3code'] not in pop_share:
        pop_share[row['countryiso3code']] = row['value'] / TOTAL_WAP

# Validação com BRA
bra_pct = df_enriched[(df_enriched['geo_id']=='BRA') & (df_enriched['variable']=='usage_pct')]['value'].values[0]
bra_idx_ref = df_enriched[(df_enriched['geo_id']=='BRA') & (df_enriched['variable']=='usage_per_capita_index')]['value'].values[0]
bra_idx_calc = bra_pct / (pop_share['BRA'] * 100)

print(f'TOTAL_WAP (denominador fixo): {TOTAL_WAP:,.0f}')
print(f'Países em pop_share: {len(pop_share)}')
print(f'\nValidação BRA Ago 2025:')
print(f'  enriched (referência):  {bra_idx_ref:.4f}')
print(f'  re-computado:           {bra_idx_calc:.4f}')
print(f'  diferença:              {abs(bra_idx_ref-bra_idx_calc)/bra_idx_ref*100:.2f}%')

## 3. Carrega todas as releases

In [ ]:
RELEASES = [
    {
        'label':        'Ago 2025',
        'file':         'release_2025_09_15/data/output/aei_enriched_claude_ai_2025-08-04_to_2025-08-11.csv',
        'reference_dt': '2025-08-07',
        'schema':       'enriched',
    },
    {
        'label':        'Nov 2025',
        'file':         'release_2026_01_15/data/intermediate/aei_raw_claude_ai_2025-11-13_to_2025-11-20.csv',
        'reference_dt': '2025-11-16',
        'schema':       'raw',
    },
    {
        'label':        'Fev 2026',
        'file':         'release_2026_03_24/data/aei_raw_claude_ai_2026-02-05_to_2026-02-12.csv',
        'reference_dt': '2026-02-08',
        'schema':       'raw',
    },
]

frames = []

for meta in RELEASES:
    print(f"\nCarregando {meta['label']} ...")
    raw = pd.read_csv(dl(meta['file']))

    # Normaliza geo_id para ISO-3 (releases raw usam ISO-2)
    if meta['schema'] == 'raw':
        raw['geo_id'] = raw['geo_id'].apply(to_iso3)

    # Metadados temporais
    raw['release_date']  = pd.Timestamp(meta['reference_dt'])
    raw['release_label'] = meta['label']
    raw['schema_type']   = meta['schema']
    raw['country_name']  = raw['geo_id'].map(country_name_map)

    # Calcula usage_per_capita_index para releases raw
    # Formula: usage_pct / (pop_share * 100)
    # usa TOTAL_WAP fixo do enriched — mesma metodologia da Anthropic
    if meta['schema'] == 'raw':
        mask = (
            (raw['geography'] == 'country') &
            (raw['facet']     == 'country') &
            (raw['variable']  == 'usage_pct')
        )
        idx_rows = raw[mask].copy()
        idx_rows['value'] = idx_rows.apply(
            lambda r: r['value'] / (pop_share.get(r['geo_id'], float('nan')) * 100),
            axis=1
        )
        idx_rows['variable'] = 'usage_per_capita_index'
        raw = pd.concat([raw, idx_rows], ignore_index=True)

    frames.append(raw)

    n_countries = raw[raw['geography'] == 'country']['geo_id'].nunique()
    print(f'  {len(raw):,} linhas | {n_countries} países')

    bra = raw[(raw['geo_id']=='BRA') & (raw['variable']=='usage_per_capita_index')]['value'].values
    print(f'  BRA index: {bra[0]:.4f}' if len(bra) > 0 else '  BRA: NAO ENCONTRADO')

df_all = pd.concat(frames, ignore_index=True)
print(f'\nDataframe consolidado: {df_all.shape}')

## 4. Brasil — série temporal

In [ ]:
bra_trend = df_all[
    (df_all['geo_id']    == 'BRA') &
    (df_all['geography'] == 'country') &
    (df_all['facet']     == 'country') &
    (df_all['variable']  == 'usage_per_capita_index')
][['release_label', 'release_date', 'value']].sort_values('release_date')

print('Brasil — usage_per_capita_index ao longo do tempo:')
print(bra_trend.to_string(index=False))
print()

# LATAM comparavel
latam = ['BRA', 'ARG', 'MEX', 'COL', 'CHL', 'PER', 'URY']
latam_trend = df_all[
    (df_all['geo_id'].isin(latam)) &
    (df_all['geography'] == 'country') &
    (df_all['facet']     == 'country') &
    (df_all['variable']  == 'usage_per_capita_index')
][['release_label', 'geo_id', 'value']].sort_values(['geo_id', 'release_label'])

print('LATAM — usage_per_capita_index por release:')
print(latam_trend.to_string(index=False))

## 5. Salva parquets

In [ ]:
out = DATA_DIR / 'aei_all_releases.parquet'
df_all.to_parquet(out, index=False)
print(f'Salvo: {out} ({out.stat().st_size/1e6:.1f} MB)')

out_aug = DATA_DIR / 'aei_aug2025.parquet'
df_all[df_all['release_label'] == 'Ago 2025'].to_parquet(out_aug, index=False)
print(f'Salvo: {out_aug} ({out_aug.stat().st_size/1e6:.1f} MB)')

out_feb = DATA_DIR / 'aei_feb2026.parquet'
df_all[df_all['release_label'] == 'Fev 2026'].to_parquet(out_feb, index=False)
print(f'Salvo: {out_feb} ({out_feb.stat().st_size/1e6:.1f} MB)')

print('\nPara carregar em outros notebooks:')
print("  df_all = pd.read_parquet('../data/aei_all_releases.parquet')")
print("  df_aug = pd.read_parquet('../data/aei_aug2025.parquet')")
print("  df_feb = pd.read_parquet('../data/aei_feb2026.parquet')")

## 6. Checklist de qualidade

In [ ]:
bra_ago_idx = float(df_all[
    (df_all['geo_id']=='BRA') &
    (df_all['release_label']=='Ago 2025') &
    (df_all['variable']=='usage_per_capita_index')
]['value'].values[0])

checks = [
    ('3 releases presentes',
     df_all['release_label'].nunique() == 3),
    ('Brasil presente nas 3 releases',
     df_all[df_all['geo_id'] == 'BRA']['release_label'].nunique() == 3),
    ('country_name preenchida para >80% dos paises',
     df_all[df_all['geography'] == 'country']['country_name'].notna().mean() > 0.8),
    ('usage_per_capita_index nas 3 releases',
     df_all[df_all['variable'] == 'usage_per_capita_index']['release_label'].nunique() == 3),
    ('BRA index Ago 2025 entre 0.85 e 1.05 (validacao metodologica)',
     0.85 <= bra_ago_idx <= 1.05),
    ('automation_pct disponivel em Ago 2025',
     len(df_all[(df_all['release_label']=='Ago 2025') & (df_all['variable']=='automation_pct')]) > 0),
    ('human_only_ability_pct disponivel em Fev 2026',
     len(df_all[(df_all['release_label']=='Fev 2026') & (df_all['variable']=='human_only_ability_pct')]) > 0),
    ('Parquet completo salvo',
     (DATA_DIR / 'aei_all_releases.parquet').exists()),
]

all_pass = True
for label, result in checks:
    icon = 'OK' if result else 'FALHOU'
    print(f'  [{icon}] {label}')
    if not result:
        all_pass = False

print()
if all_pass:
    print('Setup completo! Pode rodar os demais notebooks.')
else:
    print('Alguns checks falharam.')